In [ ]:
import argparse
import re
import shlex
import itertools
import pandas as pd
import hydra
from hydra.core.global_hydra import GlobalHydra
import torch
import time       
import datetime   

# Import run to trigger OmegaConf resolvers
import topobench.run as tb_run 
from topobench.data.preprocessor import PreProcessor
from topobench.dataloader import TBDataloader

def count_detailed_parameters(model: torch.nn.Module, only_trainable: bool = True):
    """Counts parameters and groups them by their top-level module name."""
    total_params = 0
    breakdown = {"Backbone": 0, "Feature_Encoder": 0, "Readout": 0, "Other": 0}

    for name, p in model.named_parameters():
        if only_trainable and not p.requires_grad:
            continue
        
        num = p.numel()
        total_params += num
        
        if "backbone" in name and "wrapper" not in name:
            breakdown["Backbone"] += num
        elif "feature_encoder" in name:
            breakdown["Feature_Encoder"] += num
        elif "readout" in name:
            breakdown["Readout"] += num
        else:
            breakdown["Other"] += num
            
    return total_params, breakdown

def get_override_mapping(content: str) -> dict[str, str]:
    """Parses the pipe-separated SWEEP_CONFIG block from the bash script."""
    
    block_pattern = r"SWEEP_CONFIG=\((.*?)\n\s*\)"
    block_match = re.search(block_pattern, content, re.DOTALL)
    
    if not block_match:
        raise ValueError("Could not find the SWEEP_CONFIG block. Check if it ends with a ')' on a new line.")
    
    sweep_body = block_match.group(1)
    mapping_pattern = r"\"[^|]*\|([^|]+)\|\$\{([^|*\[]+)"
    pairs = re.findall(mapping_pattern, sweep_body)

    return {bash_arr.strip(): hydra_key.strip() for hydra_key, bash_arr in pairs}

def parse_bash_array(content: str, array_name: str, mapping: dict) -> tuple[list[str], str]:
    """Extracts elements from a bash array and its corresponding Hydra override string."""
    pattern = rf"{array_name}=\((.*?)\)"
    match = re.search(pattern, content, re.DOTALL)
    
    if not match:
        raise ValueError(f"Could not find array '{array_name}' in the script.")
    
    items = shlex.split(match.group(1), comments=True)
    override_key = mapping.get(array_name, array_name)
    
    return items, override_key

def get_hydra_val(val: str) -> str:
    """Extracts the hydra value from the 'alias::hydra_value' syntax."""
    return val.split('::', 1)[1] if '::' in val else val

def main(sh_path: str, sweep_arrays: list[str] = None, out_path: str = None):
    start_time_overall = time.time()
    
    if sweep_arrays is None:
        sweep_arrays = ["models", "neighborhoods", "encodings", "num_layers", "hidden_channels"]
        
    print(f"Parsing parameters from {sh_path}...")
    
    with open(sh_path, 'r') as f:
        content = f.read()
        
    mapping = get_override_mapping(content)
    
    sweep_data = {}
    for arr in sweep_arrays:
        vals_raw, override_key = parse_bash_array(content, arr, mapping)
        sweep_data[arr] = {
            "override_key": override_key,
            "values": [get_hydra_val(v) for v in vals_raw]
        }
        
    fixed_params = {}
    fixed_overrides = []
    
    print("\n--- Fixed Parameters (Using 1st value from script) ---")
    for arr_name, override_key in mapping.items():
        if arr_name not in sweep_arrays:
            try:
                vals_raw, _ = parse_bash_array(content, arr_name, mapping)
                first_val = get_hydra_val(vals_raw[0])
                fixed_params[arr_name] = first_val
                fixed_overrides.append(f"{override_key}={first_val}")
                print(f"Fixed '{arr_name}': {first_val}")
            except Exception as e:
                print(f"Warning: Could not set fixed value for '{arr_name}'")
    print("-" * 54 + "\n")
    
    keys = list(sweep_data.keys())
    value_lists = [sweep_data[k]["values"] for k in keys]
    combinations = list(itertools.product(*value_lists))
    
    results = []
    print(f"Total combinations to evaluate: {len(combinations)}")
    print("-" * 60)

    GlobalHydra.instance().clear()
    hydra.initialize(version_base="1.3", config_path="../configs") 

    for idx, combo in enumerate(combinations):
        # --- START ITERATION TIMER ---
        iter_start_time = time.time()
        
        combo_dict = dict(zip(keys, combo))
        
        model_val = combo_dict.get("models", "")
        dataset_val = combo_dict.get("datasets", fixed_params.get("datasets", ""))
        
        if model_val.startswith('cell/') and dataset_val.startswith('simplicial/'):
            print(f"[{idx+1}/{len(combinations)}] Skipping invalid combo (Cell model + Simplicial dataset)")
            continue

        overrides = []
        
        for k, v in combo_dict.items():
            override_key = sweep_data[k]['override_key']
            
            if "@@@" in str(v):
                parts = str(v).split("@@@")
                primary_val = parts[0].strip()
                overrides.append(f"{override_key}={primary_val}")
                
                for extra_override in parts[1:]:
                    if extra_override.strip():
                        overrides.append(extra_override.strip())
            else:
                overrides.append(f"{override_key}={v}")
        
        for arr_name, override_key in mapping.items():
            if arr_name not in sweep_arrays:
                fixed_val = fixed_params.get(arr_name, "")
                if fixed_val:
                    if "@@@" in str(fixed_val):
                        parts = str(fixed_val).split("@@@")
                        overrides.append(f"{override_key}={parts[0].strip()}")
                        for extra_override in parts[1:]:
                            if extra_override.strip():
                                overrides.append(extra_override.strip())
                    else:
                        overrides.append(f"{override_key}={fixed_val}")
            
        overrides.extend([
            "train=False", 
            "test=False",
            "trainer.accelerator=cpu",
            "trainer.devices=auto",
            "dataset.dataloader_params.batch_size=1"
        ])

        try:
            cfg = hydra.compose(config_name="run.yaml", overrides=overrides)
            
            dataset_loader = hydra.utils.instantiate(cfg.dataset.loader)
            loaded_dataset, dataset_dir = dataset_loader.load()
            
            transform_config = hydra.utils.instantiate(cfg.transforms) if cfg.get("transforms", None) else None
            
            preprocessor = PreProcessor(loaded_dataset, dataset_dir, transform_config)
            dataset_train, dataset_val, dataset_test = preprocessor.load_dataset_splits(cfg.dataset.split_params)
            
            datamodule = TBDataloader(
                dataset_train=dataset_train,
                dataset_val=dataset_val,
                dataset_test=dataset_test,
                **cfg.dataset.get("dataloader_params", {}),
            )
            
            instantiated_model = hydra.utils.instantiate(
                cfg.model,
                evaluator=cfg.evaluator,
                optimizer=cfg.optimizer,
                loss=cfg.loss,
            )
            
            total_params, breakdown = count_detailed_parameters(instantiated_model)
            
            res = {k: combo_dict[k] for k in keys} 
            res.update({
                "Total_Params": total_params,
                "Backbone_Params": breakdown["Backbone"],
                "Encoder_Params": breakdown["Feature_Encoder"],
                "Readout_Params": breakdown["Readout"],
                "Other_Params": breakdown["Other"]
            })
            results.append(res)
            
            # --- END ITERATION TIMER (SUCCESS) ---
            iter_time = time.time() - iter_start_time
            
            log_info = " ".join([f"{k[:3]}:{str(v).split('/')[-1]}" for k, v in combo_dict.items()])
            print(f"[{idx+1}/{len(combinations)}] SUCCESS in {iter_time:.2f}s | Total: {total_params:,} ({log_info})")

        except Exception as e:
            # --- END ITERATION TIMER (FAIL) ---
            iter_time = time.time() - iter_start_time
            print(f"[{idx+1}/{len(combinations)}] FAILED in {iter_time:.2f}s | Error: {e}")

    print("-" * 60)
    df = pd.DataFrame(results)
    
    if not df.empty and out_path is not None:
        df = df.sort_values(by=[keys[0], "Backbone_Params"])
        df.to_csv(out_path, index=False)
        print(f"Saved results to {out_path}")
    else:
        print("No successful combinations to report.")

    end_time_overall = time.time()
    total_time_seconds = end_time_overall - start_time_overall
    formatted_time = str(datetime.timedelta(seconds=int(total_time_seconds)))
    
    print("-" * 60)
    print(f"Overall Script Time: {formatted_time} (HH:MM:SS)")
    print("-" * 60)

In [ ]:
sh_path = "../scripts/hopse_m.sh"
out_path = "parameter_sweep_results_hopse_m.csv"
# You can now easily pass arbitrary combinations from the bash script!
target_arrays = ["models", "encodings", "num_layers", "hidden_channels"]
main(sh_path, target_arrays, out_path)

In [2]:
sh_path = "../scripts/gcn.sh"
out_path = "parameter_sweep_results_gcn.csv"
# You can now easily pass arbitrary combinations from the bash script!
target_arrays = ["models", "transform_presets", "num_layers", "hidden_channels"]
main(sh_path, target_arrays, out_path)

Parsing parameters from ../scripts/gcn.sh...

--- Fixed Parameters (Using 1st value from script) ---
Fixed 'datasets': graph/cocitation_cora
Fixed 'proj_dropouts': 0.25
Fixed 'lrs': 0.01
Fixed 'weight_decays': 0
Fixed 'batch_sizes': 128
Fixed 'DATA_SEEDS': 0
------------------------------------------------------

Total combinations to evaluate: 18
------------------------------------------------------------
[1/18] SUCCESS | Total: 1,005,778 (mod:gcn tra:no_transform num:1 hid:512)
[2/18] SUCCESS | Total: 2,531,538 (mod:gcn tra:no_transform num:1 hid:1024)
[3/18] SUCCESS | Total: 1,268,434 (mod:gcn tra:no_transform num:2 hid:512)
[4/18] SUCCESS | Total: 3,581,138 (mod:gcn tra:no_transform num:2 hid:1024)
[5/18] SUCCESS | Total: 1,793,746 (mod:gcn tra:no_transform num:4 hid:512)
[6/18] SUCCESS | Total: 5,680,338 (mod:gcn tra:no_transform num:4 hid:1024)


Processing...



Applying transforms to 1 graphs...


Processing graphs:   0%|          | 0/1 [00:00<?, ?graph/s]


--- LapPE Debug Report ---
Exact compute time:  0.1238s
Fast compute time:   0.0498s
Speedup Factor:      2.49x
Mean Abs Error:      5.804226e-03
--------------------------


--- RWSE Debug Report (Nodes: 2708, Edges: 10556) ---
Method     | Status (Time | Peak VRAM)     
---------------------------------------------
Dense      | 0.1230s | 120.24 MB           
Sparse     | 0.1296s | 824.06 MB           
Batched    | 0.0242s | 85.08 MB            
---------------------------------------------
Max Abs Error (Dense vs Sparse):   1.490116e-07
Max Abs Error (Sparse vs Batched): 1.192093e-07
---------------------------------------------


--- ElectrostaticPE Debug Report ---


KeyboardInterrupt: 